In [15]:
# ============================================================
# CELL 1: IMPORT LIBRARIES
# Mục đích: Import các thư viện sử dụng trong preprocessing
# ============================================================

# Pandas: Xử lý dữ liệu dạng bảng
import pandas as pd

# NumPy: Hỗ trợ tính toán số học
import numpy as np

# Path: Quản lý đường dẫn file
from pathlib import Path

# train_test_split: Chia dữ liệu thành tập train và test
from sklearn.model_selection import train_test_split

# StandardScaler: Chuẩn hóa dữ liệu số
from sklearn.preprocessing import StandardScaler

# OneHotEncoder: Mã hóa dữ liệu phân loại thành các cột số
from sklearn.preprocessing import OneHotEncoder

# ColumnTransformer: Áp dụng các phương pháp xử lý
# khác nhau cho từng nhóm cột
from sklearn.compose import ColumnTransformer

# Kiểm tra phiên bản Pandas và Scikit-learn
import sklearn

print("Pandas version:", pd.__version__)
print("Scikit-learn version:", sklearn.__version__)

print("Libraries imported successfully!")

Pandas version: 3.1.1
Scikit-learn version: 1.9.1
Libraries imported successfully!


In [16]:
# ============================================================
# CELL 2: LOAD DATASET
# Mục đích: Đọc dữ liệu gốc để bắt đầu preprocessing
# ============================================================

# Xác định thư mục gốc dự án
PROJECT_DIR = Path.cwd().parent

# Xác định đường dẫn dataset
DATA_PATH = (
    PROJECT_DIR
    / "dataset"
    / "WA_Fn-UseC_-HR-Employee-Attrition.csv"
)

# Kiểm tra dataset có tồn tại hay không
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Không tìm thấy dataset: {DATA_PATH}"
    )

# Đọc file CSV
df = pd.read_csv(DATA_PATH)

# Hiển thị thông tin cơ bản
print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

# Hiển thị 5 dòng đầu
display(df.head())

Dataset loaded successfully!
Dataset shape: (1470, 35)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [17]:
# ============================================================
# CELL 3: INITIAL DATA VALIDATION
# Mục đích: Kiểm tra chất lượng dataset trước preprocessing
# ============================================================

# Kiểm tra số dòng và số cột
print("Dataset shape:", df.shape)

# Kiểm tra tổng số giá trị thiếu
print("\nTotal missing values:")
print(df.isnull().sum().sum())

# Kiểm tra tổng số dòng trùng
print("\nTotal duplicate rows:")
print(df.duplicated().sum())

# Kiểm tra phân bố biến mục tiêu
print("\nAttrition distribution:")
print(df["Attrition"].value_counts())

Dataset shape: (1470, 35)

Total missing values:
0

Total duplicate rows:
0

Attrition distribution:
Attrition
No     1233
Yes     237
Name: count, dtype: int64


In [18]:
# ============================================================
# CELL 4: REMOVE UNNECESSARY COLUMNS
# Mục đích: Loại bỏ ID và những cột không có thông tin
# hữu ích cho việc dự đoán
# ============================================================

# Tạo bản sao dataset
# Không xử lý trực tiếp dataset gốc
df_clean = df.copy()

# Danh sách các cột cần loại bỏ
columns_to_drop = [
    "EmployeeNumber",  # ID nhân viên
    "EmployeeCount",   # Giá trị cố định
    "Over18",          # Giá trị cố định
    "StandardHours"    # Giá trị cố định
]

# Kiểm tra cột tồn tại trước khi xóa
existing_columns_to_drop = [
    col for col in columns_to_drop
    if col in df_clean.columns
]

# Xóa các cột không cần thiết
df_clean = df_clean.drop(
    columns=existing_columns_to_drop
)

# Hiển thị những cột đã loại bỏ
print("Removed columns:")
print(existing_columns_to_drop)

# Hiển thị kích thước dữ liệu sau khi loại bỏ cột
print("\nDataset shape after removing columns:")
print(df_clean.shape)

# Kiểm tra 5 dòng đầu tiên
display(df_clean.head())

Removed columns:
['EmployeeNumber', 'EmployeeCount', 'Over18', 'StandardHours']

Dataset shape after removing columns:
(1470, 31)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,2,Female,...,3,1,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,3,Male,...,4,4,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,4,Male,...,3,2,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,4,Female,...,3,3,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,Male,...,3,4,1,6,3,3,2,2,2,2


In [19]:
# ============================================================
# CELL 5: SPLIT FEATURES AND TARGET
# Mục đích: Tách dữ liệu đầu vào và biến cần dự đoán
# ============================================================

# Xác định tên cột mục tiêu
TARGET = "Attrition"

# X chứa toàn bộ đặc trưng đầu vào
# Loại bỏ cột Attrition khỏi X
X = df_clean.drop(columns=[TARGET])

# y chứa nhãn nghỉ việc
y = df_clean[TARGET].copy()

# Hiển thị kích thước của X và y
print("Features shape:", X.shape)
print("Target shape:", y.shape)

# Hiển thị danh sách features
print("\nFeature columns:")
print(X.columns.tolist())

# Hiển thị phân bố target
print("\nTarget distribution:")
print(y.value_counts())

Features shape: (1470, 30)
Target shape: (1470,)

Feature columns:
['Age', 'BusinessTravel', 'DailyRate', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'OverTime', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']

Target distribution:
Attrition
No     1233
Yes     237
Name: count, dtype: int64


In [20]:
# ============================================================
# CELL 6: ENCODE TARGET VARIABLE
# Mục đích: Chuyển Attrition từ Yes/No sang 0/1
# ============================================================

# Ánh xạ:
# No  = 0: Nhân viên không nghỉ việc
# Yes = 1: Nhân viên nghỉ việc

target_mapping = {
    "No": 0,
    "Yes": 1
}

# Chuyển target từ chuỗi thành số
y = y.map(target_mapping)

# Kiểm tra dữ liệu sau mã hóa
print("Encoded target distribution:")
print(y.value_counts())

# Kiểm tra có xuất hiện giá trị không hợp lệ hay không
print("\nUnique target values:")
print(y.unique())

# Kiểm tra giá trị thiếu sau encoding
print("\nMissing target values:")
print(y.isnull().sum())

Encoded target distribution:
Attrition
0    1233
1     237
Name: count, dtype: int64

Unique target values:
[1 0]

Missing target values:
0


In [21]:
# ============================================================
# CELL 7: TRAIN-TEST SPLIT
# Mục đích: Chia dataset thành tập huấn luyện và tập kiểm tra
# ============================================================

# Chia dữ liệu:
# 80% dùng để huấn luyện
# 20% dùng để kiểm tra
#
# random_state=42:
# Giúp kết quả chia dữ liệu có thể tái lập
#
# stratify=y:
# Giữ tỷ lệ Attrition 0 và 1 gần giống nhau
# ở cả train và test

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Kiểm tra kích thước sau khi chia
print("Training set:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nTesting set:")
print("X_test:", X_test.shape)
print("y_test:", y_test)

# Kiểm tra tỷ lệ nghỉ việc trên training set
print("\nTraining target distribution (%):")
print((y_train.value_counts(normalize=True) * 100).round(2))

# Kiểm tra tỷ lệ nghỉ việc trên testing set
print("\nTesting target distribution (%):")
print((y_test.value_counts(normalize=True) * 100).round(2))

Training set:
X_train: (1176, 30)
y_train: (1176,)

Testing set:
X_test: (294, 30)
y_test: 1061    0
891     0
456     0
922     0
69      1
       ..
1269    0
1352    0
1236    1
1023    0
285     0
Name: Attrition, Length: 294, dtype: int64

Training target distribution (%):
Attrition
0    83.84
1    16.16
Name: proportion, dtype: float64

Testing target distribution (%):
Attrition
0    84.01
1    15.99
Name: proportion, dtype: float64


In [22]:
# ============================================================
# CELL 8: IDENTIFY FEATURE TYPES
# Mục đích: Phân loại các features để áp dụng preprocessing
# phù hợp cho từng loại dữ liệu
# ============================================================

# Lấy các cột dữ liệu phân loại
categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

# Lấy các cột dữ liệu số
numerical_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

# Hiển thị danh sách cột phân loại
print("Categorical features:")
print(categorical_features)

# Hiển thị danh sách cột số
print("\nNumerical features:")
print(numerical_features)

# Đếm số lượng từng loại feature
print("\nNumber of categorical features:", len(categorical_features))
print("Number of numerical features:", len(numerical_features))

# Kiểm tra tổng số features
print(
    "\nTotal features:",
    len(categorical_features) + len(numerical_features)
)

Categorical features:
['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']

Numerical features:
['Age', 'DailyRate', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']

Number of categorical features: 7
Number of numerical features: 23

Total features: 30


C:\Users\WINDOWS\AppData\Local\Temp\ipykernel_19984\593092545.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(


In [23]:
# ============================================================
# CELL 9: BUILD PREPROCESSING PIPELINE
# Mục đích:
# - OneHotEncode dữ liệu phân loại
# - StandardScale dữ liệu số
# - Kết hợp các bước preprocessing
# ============================================================

# Tạo OneHotEncoder
# handle_unknown="ignore":
# Nếu dữ liệu test xuất hiện category chưa có trong train,
# encoder sẽ không báo lỗi.
#
# sparse_output=False:
# Trả kết quả dạng array thông thường thay vì sparse matrix.
# Dataset nhỏ nên dùng dense array là phù hợp.

categorical_transformer = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

# Tạo StandardScaler
# StandardScaler sẽ chuẩn hóa dữ liệu numerical
numerical_transformer = StandardScaler()

# ColumnTransformer:
# Áp dụng OneHotEncoder cho categorical features
# Áp dụng StandardScaler cho numerical features
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            categorical_transformer,
            categorical_features
        ),
        (
            "numerical",
            numerical_transformer,
            numerical_features
        )
    ],
    remainder="drop"
)

print("Preprocessing pipeline created successfully!")
print(preprocessor)

Preprocessing pipeline created successfully!
ColumnTransformer(transformers=[('categorical',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['BusinessTravel', 'Department',
                                  'EducationField', 'Gender', 'JobRole',
                                  'MaritalStatus', 'OverTime']),
                                ('numerical', StandardScaler(),
                                 ['Age', 'DailyRate', 'DistanceFromHome',
                                  'Education', 'EnvironmentSatisfaction',
                                  'HourlyRate', 'JobInvolvement', 'JobLevel',
                                  'JobSatisfaction', 'MonthlyIncome',
                                  'MonthlyRate', 'NumCompaniesWorked',
                                  'PercentSalaryHike', 'PerformanceRating',
                                  'RelationshipSatisfac

In [24]:
# ============================================================
# CELL 10: FIT AND TRANSFORM DATA
# Mục đích:
# - Học cách xử lý từ training data
# - Chuyển training data sang dạng số
# - Chuyển testing data bằng cùng bộ preprocessing
# ============================================================

# Fit preprocessing trên training set
# Đồng thời transform training set
X_train_processed = preprocessor.fit_transform(X_train)

# Transform testing set
# KHÔNG fit lại scaler/encoder trên testing data
X_test_processed = preprocessor.transform(X_test)

# Hiển thị kích thước dữ liệu sau preprocessing
print("Processed training data shape:")
print(X_train_processed.shape)

print("\nProcessed testing data shape:")
print(X_test_processed.shape)

print("\nPreprocessing completed successfully!")

Processed training data shape:
(1176, 51)

Processed testing data shape:
(294, 51)

Preprocessing completed successfully!


In [25]:
# ============================================================
# CELL 11: VALIDATE PROCESSED DATA
# Mục đích: Kiểm tra dữ liệu sau khi encoding và scaling
# ============================================================

# Kiểm tra số lượng features sau preprocessing
print("Number of features before preprocessing:", X_train.shape[1])

print(
    "Number of features after preprocessing:",
    X_train_processed.shape[1]
)

# Kiểm tra dữ liệu có giá trị thiếu hay không
print("\nMissing values in processed train:")
print(np.isnan(X_train_processed).sum())

print("\nMissing values in processed test:")
print(np.isnan(X_test_processed).sum())

# Kiểm tra giá trị vô hạn
print("\nInfinite values in train:")
print(np.isinf(X_train_processed).sum())

print("\nInfinite values in test:")
print(np.isinf(X_test_processed).sum())

# Kiểm tra dữ liệu đã được chuẩn hóa
print("\nMean of numerical features after scaling:")
print(
    X_train_processed[:, -len(numerical_features):]
    .mean(axis=0)
    .round(3)
)

print("\nStandard deviation of numerical features after scaling:")
print(
    X_train_processed[:, -len(numerical_features):]
    .std(axis=0)
    .round(3)
)

Number of features before preprocessing: 30
Number of features after preprocessing: 51

Missing values in processed train:
0

Missing values in processed test:
0

Infinite values in train:
0

Infinite values in test:
0

Mean of numerical features after scaling:
[-0.  0. -0. -0.  0.  0.  0.  0.  0. -0.  0.  0. -0. -0. -0.  0. -0.  0.
 -0.  0.  0.  0.  0.]

Standard deviation of numerical features after scaling:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [26]:
# ============================================================
# CELL 12: GET FEATURE NAMES AFTER ENCODING
# Mục đích: Lấy tên các features sau One-Hot Encoding
# ============================================================

# Lấy tên các cột sau khi OneHotEncoder
encoded_feature_names = (
    preprocessor
    .named_transformers_["categorical"]
    .get_feature_names_out(categorical_features)
    .tolist()
)

# Numerical features vẫn giữ nguyên tên
processed_feature_names = (
    encoded_feature_names
    + numerical_features
)

# Kiểm tra số lượng tên cột
print("Total processed feature names:", len(processed_feature_names))

# Hiển thị danh sách features
print("\nProcessed feature names:")

for index, feature in enumerate(processed_feature_names, start=1):
    print(f"{index}. {feature}")

Total processed feature names: 51

Processed feature names:
1. BusinessTravel_Non-Travel
2. BusinessTravel_Travel_Frequently
3. BusinessTravel_Travel_Rarely
4. Department_Human Resources
5. Department_Research & Development
6. Department_Sales
7. EducationField_Human Resources
8. EducationField_Life Sciences
9. EducationField_Marketing
10. EducationField_Medical
11. EducationField_Other
12. EducationField_Technical Degree
13. Gender_Female
14. Gender_Male
15. JobRole_Healthcare Representative
16. JobRole_Human Resources
17. JobRole_Laboratory Technician
18. JobRole_Manager
19. JobRole_Manufacturing Director
20. JobRole_Research Director
21. JobRole_Research Scientist
22. JobRole_Sales Executive
23. JobRole_Sales Representative
24. MaritalStatus_Divorced
25. MaritalStatus_Married
26. MaritalStatus_Single
27. OverTime_No
28. OverTime_Yes
29. Age
30. DailyRate
31. DistanceFromHome
32. Education
33. EnvironmentSatisfaction
34. HourlyRate
35. JobInvolvement
36. JobLevel
37. JobSatisfaction


In [27]:
# ============================================================
# CELL 13: CONVERT PROCESSED DATA TO DATAFRAME
# Mục đích: Giúp kiểm tra dữ liệu đã preprocessing dễ dàng
# ============================================================

# Chuyển training data thành DataFrame
X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=processed_feature_names,
    index=X_train.index
)

# Chuyển testing data thành DataFrame
X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=processed_feature_names,
    index=X_test.index
)

# Hiển thị 5 dòng đầu của training data sau preprocessing
print("Processed training data:")
display(X_train_processed_df.head())

# Hiển thị 5 dòng đầu của testing data sau preprocessing
print("\nProcessed testing data:")
display(X_test_processed_df.head())

Processed training data:


,BusinessTravel_Non-Travel,BusinessTravel_Travel_Frequently,BusinessTravel_Travel_Rarely,Department_Human Resources,Department_Research & Development,Department_Sales,EducationField_Human Resources,EducationField_Life Sciences,EducationField_Marketing,EducationField_Medical,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
1194,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,...,-0.432065,0.240218,2.613100,2.261482,-0.605389,0.337621,-0.665706,-0.625365,-0.368024,-0.616406
128,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,-0.432065,0.240218,0.247430,-1.072675,-0.605389,0.337621,-0.830071,-0.905635,-0.056884,-0.897047
810,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,-0.432065,1.160403,0.247430,1.492061,0.190962,0.337621,0.813578,1.336527,0.565398,1.348076
478,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,-0.432065,0.240218,-0.935405,-0.559727,-1.401740,0.337621,-0.008246,-0.064824,-0.679165,0.506155
491,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,-0.432065,-0.679966,0.247430,-0.175017,0.190962,0.337621,0.156119,0.775986,0.565398,0.786795



Processed testing data:


,BusinessTravel_Non-Travel,BusinessTravel_Travel_Frequently,BusinessTravel_Travel_Rarely,Department_Human Resources,Department_Research & Development,Department_Sales,EducationField_Human Resources,EducationField_Life Sciences,EducationField_Marketing,EducationField_Medical,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
1061,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,...,-0.432065,0.240218,0.247430,-1.329148,-0.605389,0.337621,-0.994436,-1.185905,-0.679165,-1.177687
891,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,-0.432065,1.160403,0.247430,-0.175017,1.783664,0.337621,0.484849,0.215446,1.498820,0.786795
456,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,...,2.314467,0.240218,0.247430,-0.175017,0.190962,-1.055510,-0.336976,-0.064824,-0.679165,-0.897047
922,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,-0.432065,1.160403,1.430265,1.876772,0.987313,-1.055510,2.950322,1.336527,3.676804,2.470637
69,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,-0.432065,-1.600151,0.247430,-1.200911,-2.198092,-1.055510,-0.994436,-1.185905,-0.679165,-1.177687


In [28]:
# ============================================================
# CELL 14: PREPROCESSING SUMMARY
# Mục đích: Tổng kết toàn bộ quá trình tiền xử lý dữ liệu
# ============================================================

print("=" * 60)
print("EMPLOYEE ATTRITION - PREPROCESSING SUMMARY")
print("=" * 60)

# Dataset ban đầu
print("\nOriginal dataset:")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

# Dataset sau khi loại bỏ cột không cần thiết
print("\nDataset after removing unnecessary columns:")
print("Rows:", df_clean.shape[0])
print("Columns:", df_clean.shape[1])

# Train/Test
print("\nTrain/Test split:")
print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

# Features
print("\nOriginal features:", X_train.shape[1])

# Features sau OneHotEncoding
print("Processed features:", X_train_processed.shape[1])

# Target
print("\nTarget distribution in training:")
print(y_train.value_counts())

print("\nTarget distribution in testing:")
print(y_test.value_counts())

print("\nPreprocessing completed!")
print("=" * 60)

EMPLOYEE ATTRITION - PREPROCESSING SUMMARY

Original dataset:
Rows: 1470
Columns: 35

Dataset after removing unnecessary columns:
Rows: 1470
Columns: 31

Train/Test split:
Training samples: 1176
Testing samples: 294

Original features: 30
Processed features: 51

Target distribution in training:
Attrition
0    986
1    190
Name: count, dtype: int64

Target distribution in testing:
Attrition
0    247
1     47
Name: count, dtype: int64

Preprocessing completed!
